# MannWhitneySelector

Scores each feature alone with the two-sided Mann-Whitney U test, then keeps
the k with the smallest p-values.

The U test uses only the order of a feature's values. Its U is the
feature's AUC times n1 n2: the chance that a random sample of one class
ranks above a random sample of the other.

- **Cost:** one ranking pass over the data. About 10 s on 1,500 x 28,500.
- **Strength:** no normality assumption. Any monotone transform of a
  feature, such as a log, leaves the selection unchanged.
- **Weakness:** one feature at a time, so correlated features all pass
  together. Like the F-test, it misses a difference in spread alone.
- **Two classes only.** p-values are kept as logs, so they stay finite
  below 1e-308.

In [1]:
import numpy as np
from sklearn.datasets import make_classification

from featsel.selectors import ANOVAFSelector, MannWhitneySelector

rng = np.random.default_rng(0)

# 500 features, the first 10 informative (shuffle=False keeps them first).
X, y = make_classification(n_samples=300, n_features=500, n_informative=10,
                           n_redundant=0, shuffle=False, random_state=0)
truth = set(range(10))

## Top k

In [2]:
selector = MannWhitneySelector(n_features=10).fit(X, y)
chosen = selector.get_support(indices=True)
print('chosen            :', sorted(chosen.tolist()))
print('truly informative :', len(truth & set(chosen)), 'of 10')

chosen            : [0, 4, 5, 7, 8, 76, 252, 273, 321, 417]
truly informative : 5 of 10


## Only the order matters

Apply a monotone transform to every feature, here `exp`. The U test sees
the same order, so it keeps the same features. The F-test sees new values,
stretched by the transform, and its choice changes.

In [3]:
X_exp = np.exp(X)
for cls in (MannWhitneySelector, ANOVAFSelector):
    before = set(cls(n_features=10).fit(X, y).get_support(indices=True))
    after = set(cls(n_features=10).fit(X_exp, y).get_support(indices=True))
    print(f'{cls.__name__:<20} same features after exp: {len(before & after)} of 10, '
          f'truly informative after exp: {len(truth & after)} of 10')

MannWhitneySelector  same features after exp: 10 of 10, truly informative after exp: 5 of 10
ANOVAFSelector       same features after exp: 7 of 10, truly informative after exp: 4 of 10


## Each feature's AUC

`auc_` is each feature's AUC for the second class in sorted order. 0.5
means no separation; 0 or 1 means perfect separation. Some informative
features sit near 0.5 too: they separate the classes only together with
other features, which a one-feature test cannot see.

In [4]:
auc = selector.auc_
print('informative features:', np.round(auc[:10], 2))
print('noise, first five    :', np.round(auc[10:15], 2))

informative features: [0.31 0.49 0.49 0.54 0.33 0.3  0.46 0.39 0.59 0.51]
noise, first five    : [0.51 0.51 0.51 0.54 0.49]


## Where it fails

Here the class depends on how *far* a feature is from zero, not on its sign.
Both classes are centred at zero, so neither ranks above the other, the AUC
is near 0.5, and the feature is missed.

In [5]:
x = rng.normal(size=1000)
y_far = (np.abs(x) > 1).astype(int)
X_far = np.column_stack([x, rng.normal(size=(1000, 49))])

far = MannWhitneySelector(n_features=5).fit(X_far, y_far)
rank = np.argsort(far.feature_importances_)[::-1]
print(f'AUC of the informative feature: {far.auc_[0]:.2f}')
print('rank of the one informative feature:', int(np.where(rank == 0)[0][0]) + 1, 'of 50')

AUC of the informative feature: 0.47
rank of the one informative feature: 9 of 50
